In [1]:
import pandas as pd

df = pd.read_csv('../data/phishing_url_dataset_raw.csv')
df.shape

(235795, 55)

In [2]:

URL_ONLY_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "CharContinuationRate",
    "TLDLegitimateProb",    
    "URLCharProb",         
    "TLDLength",
    "NoOfSubDomain",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "LetterRatioInURL",
    "DegitRatioInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL",
    "IsHTTPS",
]

In [3]:
# Filter the full 55-column dataset down to just 17 URL-string-only features plus the label
# Dropping everything that would require fetching a page
df_subset = df[URL_ONLY_FEATURES + ['label']]
df_subset.shape

(235795, 18)

In [5]:
# Split the filtered data into 80% training / 20% testing, keeping the same phishing/legitimate ratio between both pieces
X = df_subset.drop('label', axis=1)
y = df_subset['label']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
print(f"Random Forest CV accuracy: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")

Random Forest CV accuracy: 0.9966 (+/- 0.0002)


In [11]:
# This is an interesting finding, because it means the URL features alone are doing almost all of the work on the 45-feature set.
# Explore this?

# Check for duplicates and label correlation
print(f"Duplicates: {df_subset.duplicated().sum()}")
print("\nCorrelation with label (top 10):")
print(df_subset.corr()['label'].sort_values(ascending=False).head(10))

Duplicates: 2415

Correlation with label (top 10):
label                   1.000000
IsHTTPS                 0.609132
URLCharProb             0.469749
CharContinuationRate    0.467735
TLDLegitimateProb       0.097389
NoOfSubDomain          -0.005955
NoOfObfuscatedChar     -0.015315
NoOfAmpersandInURL     -0.034622
ObfuscationRatio       -0.041915
IsDomainIP             -0.060202
Name: label, dtype: float64


In [13]:
# The strongest correlation is isHTTPS at 0.609 - legit URLs use HTTPs more, phishing less, makes sense.
# Next strongest URLCharProb(0.470), CharContinuationRate(0.468) - URL character patters are genuinely predictive.
# Verdict: The accuracy is not from data leakage or accidental dataset-specific encoding. It is genuine findings: URL-string features alone are highly seperable in this dataset.
# We move on to full evaluation

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Train final model on full training set
rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train)
y_pred = rf_final.predict(X_test)

# Classification report
print(classification_report(y_test, y_pred, target_names=['Phishing', 'Legitimate']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Random Forest (URL-only)')
plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [9]:
import joblib

# Save the URL-only model
joblib.dump(rf_final, '../backend/app/ml/phishguard_model_url_only.pkl')

# Save the feature column names in exact order
feature_columns_url_only = X_train.columns.tolist()
joblib.dump(feature_columns_url_only, '../backend/app/ml/feature_columns_url_only.pkl')

# Verify the list
print("Feature order (save this for extractor.py FEATURE_ORDER):")
print(feature_columns_url_only)

NameError: name 'rf_final' is not defined

In [10]:
import json
from collections import Counter
from urllib.parse import urlparse

# Part A: CharContinuationRate helper
def char_continuation_rate(url: str) -> float:
    max_run = 0
    current_run = 0
    for c in url:
        if c.isalnum():
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 0
    return max_run / len(url) if len(url) > 0 else 0

# Test it
test_url = "http://evil-phish.com"
print(f"CharContinuationRate for '{test_url}': {char_continuation_rate(test_url)}")

# Part B: Extract lookup tables from training data
# Get only legitimate URLs (label == 1)
legit_urls = df[df['label'] == 1]['URL'].tolist()

# Extract all TLDs and their frequencies
tlds = []
for url in legit_urls:
    try:
        domain = urlparse(url).netloc
        tld = domain.split('.')[-1] if '.' in domain else domain
        tlds.append(tld)
    except:
        pass

tld_freq = Counter(tlds)
tld_total = sum(tld_freq.values())
tld_probs = {tld: count / tld_total for tld, count in tld_freq.items()}

print(f"\nTop 10 TLDs (by frequency):")
for tld, prob in sorted(tld_probs.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {tld}: {prob:.4f}")

# Extract all characters and their frequencies
all_chars = []
for url in legit_urls:
    all_chars.extend(url)

char_freq = Counter(all_chars)
char_total = sum(char_freq.values())
char_probs = {char: count / char_total for char, count in char_freq.items()}

print(f"\nTop 10 characters (by frequency):")
for char, prob in sorted(char_probs.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  '{char}': {prob:.4f}")

# Save both as JSON
with open('../backend/app/ml/tld_probs.json', 'w') as f:
    json.dump(tld_probs, f)

with open('../backend/app/ml/char_probs.json', 'w') as f:
    json.dump(char_probs, f)

print("\n✓ Saved tld_probs.json and char_probs.json")

CharContinuationRate for 'http://evil-phish.com': 0.23809523809523808

Top 10 TLDs (by frequency):
  com: 0.5101
  org: 0.1225
  uk: 0.0450
  net: 0.0296
  de: 0.0245
  au: 0.0193
  jp: 0.0154
  edu: 0.0138
  it: 0.0124
  nl: 0.0114

Top 10 characters (by frequency):
  'w': 0.1158
  't': 0.1016
  '.': 0.0794
  '/': 0.0735
  's': 0.0650
  'o': 0.0566
  'h': 0.0486
  'p': 0.0483
  'e': 0.0450
  'c': 0.0403

✓ Saved tld_probs.json and char_probs.json
